In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =============================================================================
# 1. NẠP DỮ LIỆU THÔ VÀ ĐỊNH NGHĨA DANH MỤC NHÂN TỐ (FACTOR DICTIONARY)
# =============================================================================
# Đọc lại file dữ liệu thô ban đầu của bạn trong file mới
df_new = pd.read_csv(csv_path)

# Tự động tìm tên các cột chính
symbol_col = next((c for c in df_new.columns if c.lower() in ['symbol', 'ticker', 'code', 'stock']), None)
date_col = next((c for c in df_new.columns if c.lower() in ['date', 'time', 'ngày']), None)
close_col = next((c for c in df_new.columns if c.lower() in ['close', 'giá đóng cửa', 'price']), None)

df_new[date_col] = pd.to_datetime(df_new[date_col])

# Định nghĩa "Bản đồ Nhân tố" cứng cho 6 mã cổ phiếu
factor_groups = {
    "Value_Portfolio": ["MBB", "MHC"],              # Định giá rẻ, tài sản lớn
    "Momentum_Portfolio": ["FRT", "VGI"],           # Động lượng giá và tăng trưởng mạnh
    "Size_Quality_Portfolio": ["HAH", "KDH"]        # Vốn hóa vừa/nhỏ, nội tại chất lượng
}

# =============================================================================
# 2. HÀM TỰ ĐỘNG TÍNH HIỆU SUẤT CHO TỪNG PHONG CÁCH ĐẦU TƯ
# =============================================================================
def evaluate_factor_style(style_name, symbols):
    # Lọc dữ liệu theo nhóm mã được chỉ định
    df_style = df_new[df_new[symbol_col].isin(symbols)].copy()
    
    # Tạo ma trận giá và tính return làm sạch hoàn toàn (Chống lỗi NaN)
    price_matrix = df_style.pivot(index=date_col, columns=symbol_col, values=close_col).sort_index()
    returns_matrix = price_matrix.ffill().pct_change().fillna(0)
    
    # Giả định phân bổ tỷ trọng đều (Equal Weights) cho các mã trong nhóm nhân tố
    n_assets = len(symbols)
    weights = np.repeat(1 / n_assets, n_assets)
    
    # Tính chuỗi lợi nhuận tích lũy của phong cách này
    portfolio_returns = returns_matrix.dot(weights)
    cumulative_growth = (1 + portfolio_returns).cumprod() * 1000  # Quy về vốn gốc 1 tỷ (1,000 triệu)
    
    return cumulative_growth

# =============================================================================
# 3. CHẠY MÔ PHỎNG VÀ TRỰC QUAN HÓA SO SÁNH CÁC NHÂN TỐ
# =============================================================================
plt.figure(figsize=(12, 6))

# Chạy vòng lặp tính toán và vẽ đường tổng tài sản của từng phong cách
for style, symbols in factor_groups.items():
    cum_wealth = evaluate_factor_style(style, symbols)
    
    # Đặt tên hiển thị tiếng Việt chuyên nghiệp cho đồ thị
    display_name = style.replace("_", " ").title()
    if style == "Value_Portfolio": display_name = "Danh mục Nhân tố Giá trị (Value: MBB, MHC)"
    if style == "Momentum_Portfolio": display_name = "Danh mục Nhân tố Động lượng (Momentum: FRT, VGI)"
    if style == "Size_Quality_Portfolio": display_name = "Danh mục Nhân tố Quy mô & Chất lượng (Size/Quality: HAH, KDH)"
        
    plt.plot(cum_wealth, label=display_name, linewidth=2)

plt.axhline(y=1000, color='black', linestyle='--', alpha=0.5, label='Vốn gốc ban đầu (1,000 triệu)')
plt.title('BƯỚC 1: SO SÁNH HIỆU SUẤT THEO TIẾP XÚC NHÂN TỐ (FACTOR EXPOSURE)', fontsize=13, fontweight='bold')
plt.xlabel('Thời gian', fontsize=11)
plt.ylabel('Giá trị tài sản danh mục (Triệu VNĐ)', fontsize=11)
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'pandas'